# Recomposer remote worker

Runs the studio's heavy jobs (speech recognition, speaker detection, separation, MMS voices) on this
machine's GPU. The studio and this notebook share one folder: the studio writes job folders into it, this
notebook serves them and writes results back.

1. In the studio: `recomp config set worker.remote_folder <that folder as your PC sees it>` (for example the
   Google Drive for Desktop path `G:\My Drive\RecompRemote`).
2. Here: set `FOLDER` to the same folder as this machine sees it, run every cell, and leave the last one running.
3. In the studio: `recomp transcript asr S3 --on remote` (or `speaker diarize S3 --on remote`).

Speaker detection needs a Hugging Face token with access to `pyannote/speaker-diarization-community-1`:
type it into `HF_TOKEN` yourself. **Never save or share this notebook with the token filled in.**

In [ ]:
# Settings
FOLDER = "/content/drive/MyDrive/RecompRemote"   # Colab + Drive; on Kaggle e.g. "/kaggle/working/RecompRemote"
MOUNT_DRIVE = True                                # Colab only
HF_TOKEN = ""                                     # only for speaker detection; type it here, never commit it

In [ ]:
import os, sys, subprocess
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        print("not on Colab: skipping the Drive mount")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
os.makedirs(FOLDER, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper", "pyannote.audio", "transformers",
                "audio-separator[gpu]", "soundfile"], check=False)

In [ ]:
SCRIPTS = {"remote_worker.py": "\"\"\"Remote worker: serve job folders from a shared folder (Drive, a bucket mount, or a local stand-in).\n\nStandalone (standard library only), so a Kaggle/Colab notebook can run it next to the other scripts.\n\n    python remote_worker.py --folder /content/drive/MyDrive/RecompRemote [--python python3] [--once]\n\nProtocol, one writer per file:\n  jobs/<id>/job.json        written by the studio: {\"script\": \"asr\", \"args\": [...], \"created\": t}\n  jobs/<id>/in/\u2026            inputs (paths in args are relative to the job folder)\n  jobs/<id>/status.json     written by the worker: {\"state\": queued|running|done|failed, \"progress\", \"message\",\n                            \"worker\", \"heartbeat\"}\n  jobs/<id>/out/\u2026           outputs\n  jobs/<id>/cancel          written by the studio to ask for a stop\n  workers/<name>.json       worker heartbeat {\"name\", \"heartbeat\", \"host\"}\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport socket\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\n\n\ndef write_json(path: Path, data: dict) -> None:\n    tmp = path.with_suffix(\".tmp\")\n    tmp.write_text(json.dumps(data), encoding=\"utf-8\")\n    os.replace(tmp, path)\n\n\ndef read_json(path: Path) -> dict | None:\n    try:\n        return json.loads(path.read_text(encoding=\"utf-8\"))\n    except (FileNotFoundError, json.JSONDecodeError):\n        return None\n\n\ndef claim(job: Path, name: str) -> bool:\n    st = read_json(job / \"status.json\")\n    if st and st.get(\"state\") in (\"running\", \"done\", \"failed\"):\n        if st.get(\"state\") == \"running\" and time.time() - st.get(\"heartbeat\", 0) > 180:\n            pass  # its worker died: take it over\n        else:\n            return False\n    write_json(job / \"status.json\", {\"state\": \"running\", \"progress\": 0, \"worker\": name, \"heartbeat\": time.time(), \"message\": \"claimed\"})\n    time.sleep(0.5)\n    st = read_json(job / \"status.json\")\n    return bool(st and st.get(\"worker\") == name)\n\n\ndef run_job(job: Path, name: str, python: str) -> None:\n    spec = read_json(job / \"job.json\") or {}\n    script = HERE / f\"{spec.get('script')}.py\"\n    (job / \"out\").mkdir(exist_ok=True)\n    log = open(job / \"log.txt\", \"a\", encoding=\"utf-8\")\n    env = dict(os.environ, PYTHONUNBUFFERED=\"1\", PYTHONIOENCODING=\"utf-8\", **(spec.get(\"env\") or {}))\n    proc = subprocess.Popen([python, str(script), *spec.get(\"args\", [])], cwd=str(job), stdout=subprocess.PIPE,\n                            stderr=log, env=env, text=True, encoding=\"utf-8\", errors=\"replace\")\n    progress, message = 0.0, \"\"\n    last_beat = 0.0\n    assert proc.stdout is not None\n    for line in proc.stdout:\n        try:\n            msg = json.loads(line)\n            progress = float(msg.get(\"progress\", progress))\n            message = str(msg.get(\"message\", message))\n        except (ValueError, TypeError):\n            log.write(line)\n        if time.time() - last_beat > 2:\n            last_beat = time.time()\n            write_json(job / \"status.json\", {\"state\": \"running\", \"progress\": progress, \"message\": message, \"worker\": name,\n                                             \"heartbeat\": last_beat})\n        if (job / \"cancel\").exists():\n            proc.kill()\n            write_json(job / \"status.json\", {\"state\": \"failed\", \"message\": \"cancelled\", \"worker\": name, \"heartbeat\": time.time()})\n            return\n    code = proc.wait()\n    log.close()\n    state = \"done\" if code == 0 else \"failed\"\n    tail = \"\"\n    if code:\n        try:\n            tail = (job / \"log.txt\").read_text(encoding=\"utf-8\", errors=\"replace\")[-800:]\n        except OSError:\n            pass\n    write_json(job / \"status.json\", {\"state\": state, \"progress\": 1.0 if code == 0 else progress, \"message\": tail or \"done\",\n                                     \"worker\": name, \"heartbeat\": time.time(), \"exit\": code})\n\n\ndef serve(folder: Path, python: str, once: bool = False, idle_exit: float = 0.0) -> int:\n    name = f\"{socket.gethostname()}-{os.getpid()}\"\n    (folder / \"jobs\").mkdir(parents=True, exist_ok=True)\n    (folder / \"workers\").mkdir(parents=True, exist_ok=True)\n    done = 0\n    idle_since = time.time()\n    while True:\n        write_json(folder / \"workers\" / f\"{name}.json\", {\"name\": name, \"heartbeat\": time.time(), \"host\": socket.gethostname()})\n        ran = False\n        for job in sorted((folder / \"jobs\").iterdir()):\n            if not (job / \"job.json\").exists() or (job / \"cancel\").exists():\n                continue\n            if claim(job, name):\n                print(f\"[{time.strftime('%H:%M:%S')}] running {job.name}\", flush=True)\n                run_job(job, name, python)\n                done += 1\n                ran = True\n                break\n        if ran:\n            idle_since = time.time()\n            continue\n        if once or (idle_exit and time.time() - idle_since > idle_exit):\n            return done\n        time.sleep(3)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--folder\", required=True)\n    ap.add_argument(\"--python\", default=sys.executable)\n    ap.add_argument(\"--once\", action=\"store_true\")\n    ap.add_argument(\"--idle-exit\", type=float, default=0.0)\n    a = ap.parse_args()\n    n = serve(Path(a.folder), a.python, a.once, a.idle_exit)\n    print(f\"served {n} job(s)\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "asr.py": "\"\"\"Speech recognition, run under the *worker python* (faster-whisper or a Hugging Face CTC model).\n\nStandalone on purpose: it imports nothing from ``recomp`` so it runs in any environment that has\nfaster-whisper (and optionally transformers). Progress is printed as JSON lines on stdout.\n\n    python asr.py --wav audio16k.wav --out result.json [--model REPO_OR_PATH] [--language en]\n                  [--ranges 10-70,300-360] [--threads 0] [--compute-type int8] [--engine whisper|ctc]\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport sys\nimport time\nimport wave\n\nimport numpy as np\n\nSR = 16000\n\n\ndef say(**kw):\n    sys.stdout.write(json.dumps(kw, ensure_ascii=False) + \"\\n\")\n    sys.stdout.flush()\n\n\ndef load_wav(path: str, start: float = 0.0, end: float | None = None) -> np.ndarray:\n    with wave.open(path, \"rb\") as w:\n        sr = w.getframerate()\n        n = w.getnframes()\n        a = int(start * sr)\n        b = n if end is None else min(n, int(end * sr))\n        w.setpos(max(0, min(a, n)))\n        raw = w.readframes(max(0, b - a))\n    if sr != SR:\n        raise SystemExit(f\"expected {SR} Hz audio, got {sr}\")\n    return np.frombuffer(raw, dtype=\"<i2\").astype(\"float32\") / 32768.0\n\n\ndef run_whisper(args, audio_parts):\n    from faster_whisper import WhisperModel\n\n    t0 = time.time()\n    model = WhisperModel(args.model, device=\"cpu\", compute_type=args.compute_type,\n                         cpu_threads=args.threads or 0, num_workers=1)\n    say(message=f\"model loaded in {time.time() - t0:.1f}s\")\n    total = sum(len(a) for _, a in audio_parts) / SR or 1.0\n    done = 0.0\n    segments = []\n    info_lang = None\n    for offset, audio in audio_parts:\n        segs, info = model.transcribe(\n            audio, language=args.language or None, word_timestamps=True, vad_filter=not args.no_vad,\n            vad_parameters={\"min_silence_duration_ms\": 500}, beam_size=args.beam, condition_on_previous_text=False,\n            temperature=0.0)\n        info_lang = info_lang or info.language\n        for s in segs:\n            words = [{\"start\": round(w.start + offset, 3), \"end\": round(w.end + offset, 3),\n                      \"text\": w.word.strip(), \"prob\": round(w.probability, 3)} for w in (s.words or []) if w.word.strip()]\n            segments.append({\"start\": round(s.start + offset, 3), \"end\": round(s.end + offset, 3),\n                             \"text\": s.text.strip(), \"words\": words,\n                             \"conf\": round(float(np.exp(s.avg_logprob)), 3) if s.avg_logprob is not None else None})\n            say(progress=round(min(0.999, (done + s.end) / total), 4), message=f\"{s.end + offset:.0f}s\")\n        done += len(audio) / SR\n    return segments, info_lang\n\n\ndef run_ctc(args, audio_parts):\n    \"\"\"Character CTC model (e.g. badrex/Ethio-ASR-amharic): text per 20-second chunk with word times.\"\"\"\n    import torch\n    from transformers import AutoModelForCTC, AutoProcessor\n\n    proc = AutoProcessor.from_pretrained(args.model)\n    model = AutoModelForCTC.from_pretrained(args.model).eval()\n    torch.set_num_threads(max(1, args.threads or os.cpu_count() or 4))\n    segments = []\n    total = sum(len(a) for _, a in audio_parts) / SR or 1.0\n    done = 0.0\n    chunk = 20 * SR\n    for offset, audio in audio_parts:\n        for i in range(0, len(audio), chunk):\n            piece = audio[i:i + chunk]\n            if len(piece) < SR // 4:\n                continue\n            inputs = proc(piece, sampling_rate=SR, return_tensors=\"pt\")\n            with torch.no_grad():\n                logits = model(**inputs).logits[0]\n            ids = logits.argmax(-1).tolist()\n            text = proc.decode(ids).strip()\n            frame_s = len(piece) / SR / max(1, len(ids))\n            start = offset + i / SR\n            if text:\n                words = []\n                # word boundaries from the token stream: a word spans its first to last non-blank frame\n                vocab = proc.tokenizer.convert_ids_to_tokens(ids)\n                blank = proc.tokenizer.pad_token\n                delim = getattr(proc.tokenizer, \"word_delimiter_token\", \"|\")\n                cur, cur_a, cur_b = \"\", None, None\n                prev = None\n                for f, tok in enumerate(vocab):\n                    if tok == prev:\n                        cur_b = f\n                        continue\n                    prev = tok\n                    if tok == blank:\n                        continue\n                    if tok == delim:\n                        if cur:\n                            words.append({\"start\": round(start + cur_a * frame_s, 3),\n                                          \"end\": round(start + (cur_b + 1) * frame_s, 3), \"text\": cur})\n                        cur, cur_a = \"\", None\n                        continue\n                    if cur_a is None:\n                        cur_a = f\n                    cur_b = f\n                    cur += tok\n                if cur:\n                    words.append({\"start\": round(start + cur_a * frame_s, 3),\n                                  \"end\": round(start + (cur_b + 1) * frame_s, 3), \"text\": cur})\n                segments.append({\"start\": round(start, 3), \"end\": round(start + len(piece) / SR, 3),\n                                  \"text\": text, \"words\": words})\n            say(progress=round(min(0.999, (done + (i + len(piece)) / SR) / total), 4))\n        done += len(audio) / SR\n    return segments, args.language\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\"--wav\", required=True)\n    p.add_argument(\"--out\", required=True)\n    p.add_argument(\"--model\", default=\"mobiuslabsgmbh/faster-whisper-large-v3-turbo\")\n    p.add_argument(\"--engine\", default=\"whisper\", choices=[\"whisper\", \"ctc\"])\n    p.add_argument(\"--language\", default=None)\n    p.add_argument(\"--ranges\", default=\"\")\n    p.add_argument(\"--threads\", type=int, default=0)\n    p.add_argument(\"--compute-type\", default=\"int8\")\n    p.add_argument(\"--beam\", type=int, default=1)\n    p.add_argument(\"--no-vad\", action=\"store_true\")\n    args = p.parse_args()\n    parts = []\n    if args.ranges:\n        for r in args.ranges.split(\",\"):\n            a, b = (float(x) for x in r.split(\"-\"))\n            parts.append((a, load_wav(args.wav, a, b)))\n    else:\n        parts.append((0.0, load_wav(args.wav)))\n    t0 = time.time()\n    segments, lang = (run_ctc if args.engine == \"ctc\" else run_whisper)(args, parts)\n    audio_s = sum(len(a) for _, a in parts) / SR\n    result = {\"segments\": segments, \"language\": lang, \"model\": args.model, \"engine\": args.engine,\n              \"audio_s\": round(audio_s, 1), \"elapsed_s\": round(time.time() - t0, 1),\n              \"ranges\": [[o, o + len(a) / SR] for o, a in parts]}\n    tmp = args.out + \".part\"\n    with open(tmp, \"w\", encoding=\"utf-8\") as f:\n        json.dump(result, f, ensure_ascii=False)\n    os.replace(tmp, args.out)\n    say(progress=1.0, message=\"done\", segments=len(segments), elapsed_s=result[\"elapsed_s\"])\n\n\nif __name__ == \"__main__\":\n    main()\n", "diarize.py": "\"\"\"Who speaks when: pyannote speaker-diarization-community-1, run under the worker python.\n\nStandalone (imports nothing from recomp). Writes turns, exclusive turns (one speaker at a time),\nseconds per speaker and each speaker's centroid embedding (for matching people across sources).\n\n    python diarize.py --wav audio16k.wav --out diarize.json [--speakers N | --min N --max M]\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport sys\nimport time\nimport wave\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef say(**kw):\n    sys.stdout.write(json.dumps(kw) + \"\\n\")\n    sys.stdout.flush()\n\n\ndef token() -> str | None:\n    if os.environ.get(\"HF_TOKEN\"):\n        return os.environ[\"HF_TOKEN\"]\n    try:\n        from huggingface_hub import get_token\n        t = get_token()\n        if t:\n            return t\n    except Exception:  # noqa: BLE001\n        pass\n    p = Path.home() / \".cache\" / \"huggingface\" / \"token\"  # the user's own `hf auth login`\n    return p.read_text(encoding=\"utf-8\").strip() if p.exists() else None\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--wav\", required=True)\n    ap.add_argument(\"--out\", required=True)\n    ap.add_argument(\"--speakers\", type=int, default=0)\n    ap.add_argument(\"--min\", type=int, default=0)\n    ap.add_argument(\"--max\", type=int, default=0)\n    ap.add_argument(\"--pipeline\", default=\"pyannote/speaker-diarization-community-1\")\n    a = ap.parse_args()\n    os.environ[\"PYANNOTE_METRICS_ENABLED\"] = \"false\"\n    import torch\n    from pyannote.audio import Pipeline\n\n    t0 = time.time()\n    pipe = Pipeline.from_pretrained(a.pipeline, token=token())\n    if pipe is None:\n        raise SystemExit(\"could not load the diarization pipeline (gated model: accept its terms and log in once)\")\n    torch.set_num_threads(max(1, (os.cpu_count() or 4) - 2))\n    say(message=f\"pipeline loaded in {time.time() - t0:.1f}s\")\n    with wave.open(a.wav, \"rb\") as w:\n        sr = w.getframerate()\n        x = np.frombuffer(w.readframes(w.getnframes()), dtype=\"<i2\").astype(\"float32\") / 32768.0\n    kw = {}\n    if a.speakers:\n        kw[\"num_speakers\"] = a.speakers\n    else:\n        if a.min:\n            kw[\"min_speakers\"] = a.min\n        if a.max:\n            kw[\"max_speakers\"] = a.max\n\n    def hook(step_name, step_artifact, file=None, total=None, completed=None):\n        if total:\n            say(progress=round(min(0.99, completed / total), 3), message=step_name)\n\n    try:\n        res = pipe({\"waveform\": torch.from_numpy(x[None, :].copy()), \"sample_rate\": sr}, hook=hook, **kw)\n    except TypeError:\n        res = pipe({\"waveform\": torch.from_numpy(x[None, :].copy()), \"sample_rate\": sr}, **kw)\n    ann = getattr(res, \"speaker_diarization\", res)\n    excl = getattr(res, \"exclusive_speaker_diarization\", None)\n    turns = [{\"start\": round(t.start, 3), \"end\": round(t.end, 3), \"speaker\": s} for t, _, s in ann.itertracks(yield_label=True)]\n    ex = [{\"start\": round(t.start, 3), \"end\": round(t.end, 3), \"speaker\": s} for t, _, s in excl.itertracks(yield_label=True)] \\\n        if excl is not None else turns\n    labels = list(ann.labels())\n    embs = getattr(res, \"speaker_embeddings\", None)\n    centroids = {lab: [round(float(v), 6) for v in embs[i]] for i, lab in enumerate(labels)} \\\n        if embs is not None and len(embs) == len(labels) else {}\n    seconds = {lab: round(sum(t[\"end\"] - t[\"start\"] for t in ex if t[\"speaker\"] == lab), 1) for lab in labels}\n    doc = {\"pipeline\": a.pipeline, \"turns\": turns, \"exclusive\": ex, \"labels\": labels, \"centroids\": centroids,\n           \"seconds\": seconds, \"elapsed_s\": round(time.time() - t0, 1), \"audio_s\": round(len(x) / sr, 1)}\n    tmp = a.out + \".part\"\n    with open(tmp, \"w\", encoding=\"utf-8\") as f:\n        json.dump(doc, f)\n    os.replace(tmp, a.out)\n    say(progress=1.0, message=\"done\", speakers=len(labels))\n\n\nif __name__ == \"__main__\":\n    main()\n", "separate.py": "\"\"\"Vocals / background split with audio-separator (BS-RoFormer), under the worker python.\n\n    python separate.py --input body.wav --outdir DIR --models MODEL_DIR [--model model_bs_roformer_ep_317_sdr_12.9755.ckpt]\nWrites DIR/vocals.flac and DIR/background.flac.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport shutil\nimport sys\nfrom pathlib import Path\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--input\", required=True)\n    ap.add_argument(\"--outdir\", required=True)\n    ap.add_argument(\"--models\", required=True)\n    ap.add_argument(\"--model\", default=\"model_bs_roformer_ep_317_sdr_12.9755.ckpt\")\n    a = ap.parse_args()\n    from audio_separator.separator import Separator\n\n    out = Path(a.outdir)\n    tmp = out / \"_sep\"\n    tmp.mkdir(parents=True, exist_ok=True)\n    s = Separator(output_format=\"FLAC\", model_file_dir=a.models, output_dir=str(tmp),\n                  mdxc_params={\"segment_size\": 256, \"override_model_segment_size\": False, \"batch_size\": 1,\n                               \"overlap\": 2, \"pitch_shift\": 0})\n    s.load_model(a.model)\n    files = [Path(f) if os.path.isabs(f) else tmp / f for f in s.separate(a.input)]\n    vocals = next(f for f in files if \"(vocals)\" in f.name.lower())\n    other = next(f for f in files if f != vocals)\n    shutil.move(str(vocals), out / \"vocals.flac\")\n    shutil.move(str(other), out / \"background.flac\")\n    shutil.rmtree(tmp, ignore_errors=True)\n    sys.stdout.write(json.dumps({\"progress\": 1.0, \"message\": \"separated\"}) + \"\\n\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "tts_mms.py": "\"\"\"MMS-TTS (VITS) synthesis under the worker python.\n\n    python tts_mms.py --model facebook/mms-tts-orm --text \"\u2026\" --out x.wav\n    python tts_mms.py --model facebook/mms-tts-orm --batch jobs.json     # [{\"text\": \u2026, \"out\": \u2026}, \u2026]\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport wave\n\nimport numpy as np\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", required=True)\n    ap.add_argument(\"--text\")\n    ap.add_argument(\"--out\")\n    ap.add_argument(\"--batch\")\n    a = ap.parse_args()\n    import torch\n    from transformers import AutoTokenizer, VitsModel\n\n    tok = AutoTokenizer.from_pretrained(a.model)\n    model = VitsModel.from_pretrained(a.model).eval()\n    sr = model.config.sampling_rate\n    items = json.load(open(a.batch, encoding=\"utf-8\")) if a.batch else [{\"text\": a.text, \"out\": a.out}]\n    for k, it in enumerate(items):\n        with torch.no_grad():\n            wav = model(**tok(it[\"text\"], return_tensors=\"pt\")).waveform[0].numpy()\n        pcm = (np.clip(wav, -1, 1) * 32767).astype(\"<i2\")\n        with wave.open(it[\"out\"], \"wb\") as w:\n            w.setnchannels(1)\n            w.setsampwidth(2)\n            w.setframerate(sr)\n            w.writeframes(pcm.tobytes())\n        sys.stdout.write(json.dumps({\"progress\": round((k + 1) / len(items), 3)}) + \"\\n\")\n        sys.stdout.flush()\n\n\nif __name__ == \"__main__\":\n    main()\n", "selftest.py": "\"\"\"Worker self-test: copies --in to --out uppercased, reporting progress (checks the remote protocol).\"\"\"\n\nimport argparse\nimport json\nimport sys\nimport time\n\nap = argparse.ArgumentParser()\nap.add_argument(\"--in\", dest=\"inp\", required=True)\nap.add_argument(\"--out\", required=True)\na = ap.parse_args()\ntext = open(a.inp, encoding=\"utf-8\").read()\nfor k in range(3):\n    time.sleep(0.2)\n    sys.stdout.write(json.dumps({\"progress\": (k + 1) / 3, \"message\": f\"step {k + 1}\"}) + \"\\n\")\n    sys.stdout.flush()\nopen(a.out, \"w\", encoding=\"utf-8\").write(text.upper())\n"}
os.makedirs("/tmp/recomp_scripts", exist_ok=True)
for name, text in SCRIPTS.items():
    open(f"/tmp/recomp_scripts/{name}", "w", encoding="utf-8").write(text)
print("wrote", len(SCRIPTS), "worker scripts")

In [ ]:
# Serve jobs until you stop this cell. GPU is used automatically where the libraries support it.
!python /tmp/recomp_scripts/remote_worker.py --folder "$FOLDER"